# Submission 1 — Your Model, Your Beam
### ME 323 Module 1

Both class beams have been tested. The scoreboard so far:

| design | (b, H_web) | predicted | measured |
|---|---|---|---|
| equation (Pre-lab 1) | (1.25, 13.4) | 46.6 N/g | **37.64 N/g** (483.0 N) |
| GP, MUI ψ=1 (Pre-lab 2) | (1.44, 13.39) | 35.9 N/g | **38.12 N/g** (509.7 N) |

Neither beat the best beam already in the data (36.9 N/g). Now it is your
turn: fold both results in, build a model **your way**, and commit to the beam
your group will print and be graded on. There is no checkpoint for the final
answer — it is a decision, and you defend it in the memo.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120

# Fixed geometry (you choose b and H_web; everything else is set)
B, TH, L = 10.0, 18.0, 150.0     # flange width, total height, test span (mm)
LP = 172.0                        # printed length (mm); overhangs the 150 mm span
KMASS = 0.2045                    # g/mm^2: mass per unit cross-section area at 172 mm

# Handbook starting values — Pre-lab 1 calibrates the three marked ones
SY = 76e6                         # Pa, PLA strength                 (calibrated)
K_LTB = 0.33                      # fixture effective-length factor  (calibrated)
CS = 1.0                          # web shear-strength multiplier    (calibrated)
E, G = 2.5e9, 2.5e9 / 2.6         # Young's / shear modulus (Pa) — fixed
C1, C2 = 1.35, 0.55               # LTB moment-gradient / load-height factors
URL = ("https://raw.githubusercontent.com/andrewvoss8-boop/"
       "core-me-data-science-activities-public/main/data/student_beams_B10_L150.csv")
try:
    df = pd.read_csv(URL); print("loaded from GitHub")
except Exception:
    df = pd.read_csv("student_beams_B10_L150.csv"); print("loaded local copy")
df = df.rename(columns={"b_mm": "b", "H_web_mm": "H"})

def mass_g(b, H):
    A = b * H + B * (TH - H)      # cross-section area, mm^2
    return KMASS * A              # grams
df["mass_g"] = mass_g(df.b, df.H)
print(len(df), "tested beams")

new = pd.DataFrame([dict(beam_id=15, b=1.25, H=13.4, strength_N=483.0),
                    dict(beam_id=16, b=1.44, H=13.39, strength_N=509.7)])
new["mass_g"] = mass_g(new.b, new.H)
df = pd.concat([df, new], ignore_index=True)
df["str_to_weight"] = df.strength_N / df.mass_g
print(len(df), "beams. best observed:", round(df.str_to_weight.max(), 2), "N/g")

## 1. The calibrated physics (carried over from Pre-lab 1)

Provided complete this time, with the class calibration values baked in.

In [ ]:
def section_props(b, H):
    """I-section properties. b, H in mm; everything returned in METERS/SI."""
    tf = (TH - H) / 2.0
    b_, h_, B_, tf_ = b/1e3, H/1e3, B/1e3, tf/1e3
    c = (TH/1e3) / 2
    Ix = (b_*h_**3)/12 + 2*((B_*tf_**3)/12 + B_*tf_*(h_/2 + tf_/2)**2)
    Iy = (h_*b_**3)/12 + 2*(tf_*B_**3)/12
    beta = lambda t, a: 1 - 0.63*(t/a) + 0.052*(t/a)**5
    J  = (1/3)*beta(b_, h_)*h_*b_**3 + (2/3)*beta(tf_, B_)*B_*tf_**3
    Cw = Iy*(h_ + tf_)**2/4
    return dict(Ix=Ix, Iy=Iy, J=J, Cw=Cw, c=c, b=b_, h=h_)

def P_bend(p, sy):
    return 4*sy*p["Ix"] / (p["c"] * L/1e3)
def P_shear(p, sy, cs):
    return 2 * (cs*sy/np.sqrt(3)) * p["b"]*p["h"]
def P_vm(Pb, Ps):
    return 1.0/np.sqrt(1/Pb**2 + 1/Ps**2)
def P_LTB(p, sy, k):
    My = sy*p["Ix"]/p["c"]
    Lb, zg = k*L/1e3, p["c"]
    R = p["Cw"]/p["Iy"] + (Lb**2*G*p["J"])/(np.pi**2*E*p["Iy"]) + (C2*zg)**2
    Mcr = C1*np.pi**2*E*p["Iy"]/Lb**2 * (np.sqrt(R) - C2*zg)
    return 4*min(My, Mcr)/(L/1e3)
def capacity(b, H, sy, k, cs):
    p = section_props(b, H)
    return min(P_vm(P_bend(p, sy), P_shear(p, sy, cs)), P_LTB(p, sy, k))
def gov_mode(b, H, sy, k, cs):
    p = section_props(b, H)
    Pb, Ps, Pl = P_bend(p, sy), P_shear(p, sy, cs), P_LTB(p, sy, k)
    return "shear" if Ps < min(Pb, Pl) else ("LTB" if Pl < 0.999*Pb else "bend")

SY_CAL, K_CAL, CS_CAL = 6.650e+07, 0.377, 2.25   # your Pre-lab 1 calibration
print("calibrated capacity(2,12) =", round(capacity(2, 12, SY_CAL, K_CAL, CS_CAL), 1), "N")

## 2. Pick a lane: four ways to model the same 16 beams

You hold two kinds of knowledge, and they disagree in places:

- **16 real tests** — the truth, but only at 16 points, each carrying
  print-to-print noise.
- **The calibrated equations** — an answer *everywhere*, with roughly the
  right shape, but wrong in places: they over-promised the Pre-lab 1 beam by
  24%, and they contain no story at all for flange–web peeling.

A GP is a machine for the first kind. Whether — and *where* — you put the
second kind inside it is a modeling decision, and it is the decision this
section is about. The four lanes are four answers, from "no physics" to
"physics first":

| lane | inputs the GP sees | target it fits | where the physics goes |
|---|---|---|---|
| **A — plain** | (b, H) | log str/w | nowhere (the Pre-lab 2 model) |
| **B — strength target** | (b, H) | log strength [N]; divide by mass after | nowhere (different target) |
| **C — physics features** | (b, H, logP, stab) | log str/w | extra input coordinates |
| **D — residual** | (b, H) | log(measured / P_phys) | the baseline the GP corrects |

**A — plain.** The Pre-lab 2 recipe, refit on 16 beams. Its one promise:
designs with similar (b, H) get similar str/w. It contains no physics, so
between and beyond the data it has no opinion — its prediction relaxes back
toward the class-average str/w (about 33 N/g). It cannot know a region is
dangerous unless a beam already died there.

**B — strength in newtons.** The instinct is sound: mass is computed exactly
from geometry, so why make the GP learn it? Fit strength, divide by mass
yourself. The catch: strength and mass rise together — bigger sections are
both heavier and stronger — so dividing by mass *cancels* most of the trend.
Str/w spans 1.4× across these 16 beams; raw strength spans 3.0×. Lane A
quietly benefits from that cancellation. Lane B undoes it, asks 16 points to
relearn the steep strength trend, and then divides every miss by the beam's
mass — which amplifies the error most for light beams, exactly the region
that wins this contest.

**C — physics as coordinates.** A GP judges similarity with its kernel: how
far apart are two designs? Lane C changes what *far apart* means. Every
design gets two computed coordinates alongside (b, H): `logP`, the log of
the capacity the calibrated equations predict, and `stab = P_LTB / P_bend`,
how close lateral-torsional buckling is to taking over. Two designs now
count as neighbors when the physics says they should behave alike, not only
when their geometry is close. The equations act as a coordinate system, not
as a prediction.

**D — residual, physics as the first guess.** Flip the order: the calibrated
equations make the prediction everywhere, and the GP gets the smaller job of
learning the *correction factor*, log(measured / P_phys). On these 16 beams
that factor runs 0.81×–1.10× (average 0.98×). Near the data the GP bends the
physics to the measurements. Away from the data the correction relaxes back
to its average and the model becomes "physics × 0.98" — it keeps the physics
*shape* where lane A flattens to the class average.

**The bet you cannot avoid.** Lanes C and D inherit the equations' mistakes
along with their shape. The equations contain no flange–web peeling, so in a
peel-prone region with no tested beams, lane D confidently reports scaled
physics, and lane C sorts designs by a similarity that misses the point.
Putting physics in the model is a bet that the physics is more right than an
average. Sometimes it is — the next three figures open the hood on each
mechanism, and the leave-one-out scoreboard at the end puts numbers on all
four.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF

def fit_gp(data, alpha=0.03**2, feats=("b", "H"), target="log_sw"):
    """The class GP recipe: z-scored inputs, log target, RBF kernel, MLE."""
    X = data[list(feats)].values.astype(float)
    fmu, fsd = X.mean(0), X.std(0) + 1e-12
    if target == "log_sw":
        y = np.log(data.strength_N.values / mass_g(data.b.values, data.H.values))
    elif target == "log_strength":
        y = np.log(data.strength_N.values.astype(float))
    else:                                    # log residual vs calibrated physics
        Pphys = np.array([capacity(b, H, SY_CAL, K_CAL, CS_CAL)
                          for b, H in zip(data.b, data.H)])
        y = np.log(data.strength_N.values) - np.log(Pphys)
    ymean = y.mean()
    ker = C(1.0, (1e-3, 1e3)) * RBF([1.0]*X.shape[1], (1e-1, 30.0))
    gp = GaussianProcessRegressor(ker, alpha=alpha, normalize_y=False,
                                  n_restarts_optimizer=5, random_state=0)
    gp.fit((X - fmu) / fsd, y - ymean)
    return gp, fmu, fsd, ymean

def predict_sw(gp, fmu, fsd, ymean, bq, Hq, target, feats):
    Xq = build_feats(bq, Hq, feats)
    mu, sd = gp.predict((Xq - fmu)/fsd, return_std=True)
    mu = mu + ymean
    mass = mass_g(np.asarray(bq, float), np.asarray(Hq, float))
    if target == "log_sw":
        sw = np.exp(mu)
    elif target == "log_strength":
        sw = np.exp(mu)/mass
    else:
        Pphys = np.array([capacity(b, H, SY_CAL, K_CAL, CS_CAL)
                          for b, H in zip(np.atleast_1d(bq), np.atleast_1d(Hq))])
        sw = Pphys*np.exp(mu)/mass
    return sw, sd

def build_feats(bq, Hq, feats):
    bq, Hq = np.atleast_1d(np.asarray(bq, float)), np.atleast_1d(np.asarray(Hq, float))
    cols = {"b": bq, "H": Hq}
    if "logP" in feats or "stab" in feats:
        pp = [section_props(b, H) for b, H in zip(bq, Hq)]
        Pb = np.array([P_bend(p, SY_CAL) for p in pp])
        Pl = np.array([P_LTB(p, SY_CAL, K_CAL) for p in pp])
        Pv = np.array([P_vm(P_bend(p, SY_CAL), P_shear(p, SY_CAL, CS_CAL)) for p in pp])
        cols["logP"] = np.log(np.minimum(Pv, Pl))
        cols["stab"] = Pl/Pb
    return np.column_stack([cols[f] for f in feats])

LANES = {
    "A plain":    dict(feats=("b", "H"), target="log_sw"),
    "B strength": dict(feats=("b", "H"), target="log_strength"),
    "C features": dict(feats=("b", "H", "logP", "stab"), target="log_sw"),
    "D residual": dict(feats=("b", "H"), target="log_residual"),
}

def fit_lane(data, lane, alpha=0.03**2):
    cfg = LANES[lane]
    d2 = data.copy()
    Xf = build_feats(d2.b.values, d2.H.values, cfg["feats"])
    for j, f in enumerate(cfg["feats"]):
        d2[f] = Xf[:, j]
    gp, fmu, fsd, ymean = fit_gp(d2, alpha=alpha, feats=cfg["feats"], target=cfg["target"])
    return gp, fmu, fsd, ymean, cfg

print("four lanes defined:", ", ".join(LANES))

### Under the surface (i): the job each lane hands the GP

Same 16 beams, three different fitting jobs (lane C shares A's target). The
figure centers each target on its own mean, so the vertical spreads compare
fairly. Read the spread as *how much structure the GP must explain with 16
points*.

Two things to notice:

- **Lane B's job is about 3× wider.** That extra spread is the mass trend it
  volunteered to relearn.
- **A and D have nearly the same spread.** Subtracting the physics does
  *not* shrink the scatter on these beams — the correction factor varies
  about as much as str/w itself. What D changes is not the size of the
  leftover but what the model does *between and beyond* the data points.
  That is the next figure.

In [ ]:
Pphys16 = np.array([capacity(b, H, SY_CAL, K_CAL, CS_CAL) for b, H in zip(df.b, df.H)])
targets = {
    "A & C\nlog str/w":      np.log(df.str_to_weight.values),
    "B\nlog strength":       np.log(df.strength_N.values.astype(float)),
    "D\nlog(meas / P_phys)": np.log(df.strength_N.values) - np.log(Pphys16),
}
tcolors = ["tab:blue", "tab:orange", "tab:purple"]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
rows = []
for k, ((name, t), c) in enumerate(zip(targets.items(), tcolors)):
    tc = t - t.mean()
    x = k + (np.arange(len(tc)) - len(tc)/2) * 0.012   # spread points out a little
    ax.scatter(x, tc, color=c, s=28, alpha=0.85, zorder=3)
    ax.errorbar(k + 0.3, 0, yerr=tc.std(), color=c, capsize=5, lw=2)
    ax.annotate(f"±{tc.std():.3f}", (k + 0.36, tc.std()*0.45), fontsize=8, color=c)
    rows.append((name.replace("\n", "  "), tc.std(), np.exp(t.max() - t.min())))
ax.axhline(0, color="k", lw=0.6)
ax.set_xticks(range(len(targets))); ax.set_xticklabels(targets.keys())
ax.set_ylabel("target minus its own mean  [log units]")
ax.set_title("the same 16 beams, as three different GP targets")
plt.tight_layout(); plt.show()

print(f"{'target':<26}{'std (log)':>10}{'spread as a factor':>22}")
for name, s, f in rows:
    print(f"{name:<26}{s:>10.3f}{f:>18.2f}x")

### Under the surface (ii): a slice through the map at b = 1.25 mm

Both class beams live near this thin-web edge, so cut the 2-D map open
there: hold b = 1.25 mm and sweep H_web. Beams whose b is within 0.35 mm are
drawn on the slice (projected — their true b differs a little, so expect
them slightly off the curves). Bands are ±1σ.

What to look for:

1. **The dashed physics peaks near H_web ≈ 13.4, promising ≈ 46.6 N/g.**
   That is the Pre-lab 1 design; it measured 37.6. The *shape* is right —
   the good designs really are up there — but the height is not.
2. **Between beams, lane A sags back toward the class average** — no data,
   no opinion. Lane D instead rides the physics shape, scaled to match the
   nearby measurements: physics decides the shape, data decides the level.
3. **The printed kernels tell the same story.** A length scale is "how far
   you slide (in z-scored units) before the GP stops treating two designs
   as neighbors." Lane A picked a tiny H_web length scale (≈ 0.13): hug
   each beam, forget it fast — that is the wiggle you see. Lane D picked
   ≈ 0.7: once the physics has absorbed the sharp structure, the leftover
   correction is smooth.

In [ ]:
bslice = 1.25
Hline = np.linspace(5.0, 16.0, 200)
gpA, fmuA, fsdA, ymA, cfgA = fit_lane(df, "A plain")
gpD, fmuD, fsdD, ymD, cfgD = fit_lane(df, "D residual")
swA, sdA = predict_sw(gpA, fmuA, fsdA, ymA, np.full_like(Hline, bslice), Hline,
                      cfgA["target"], cfgA["feats"])
swD, sdD = predict_sw(gpD, fmuD, fsdD, ymD, np.full_like(Hline, bslice), Hline,
                      cfgD["target"], cfgD["feats"])
phys_sw = np.array([capacity(bslice, H, SY_CAL, K_CAL, CS_CAL)
                    for H in Hline]) / mass_g(bslice, Hline)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(Hline, phys_sw, "k--", lw=1.4, label="calibrated physics")
ax.plot(Hline, swA, color="tab:blue", lw=1.8, label="lane A — plain GP")
ax.fill_between(Hline, swA*np.exp(-sdA), swA*np.exp(sdA), color="tab:blue", alpha=0.15)
ax.plot(Hline, swD, color="tab:purple", lw=1.8, label="lane D — physics × GP correction")
ax.fill_between(Hline, swD*np.exp(-sdD), swD*np.exp(sdD), color="tab:purple", alpha=0.15)
near = df[np.abs(df.b - bslice) < 0.35]
ax.scatter(near.H, near.str_to_weight, c="w", edgecolor="k", s=45, zorder=5,
           label="beams with b within 0.35 mm (projected)")
ax.scatter(df.H.iloc[-2:], df.str_to_weight.iloc[-2:], c="red", marker="*",
           s=170, zorder=6, label="the two class beams")
ax.axhline(df.str_to_weight.mean(), color="gray", lw=0.8, ls=":",
           label=f"class average ({df.str_to_weight.mean():.1f} N/g)")
ax.set_xlabel("H_web [mm]"); ax.set_ylabel("str/w [N/g]")
ax.set_title(f"slice at b = {bslice} mm (bands: ±1σ)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("fitted kernels (RBF length scales in z-scored units, order = features):")
print(f"  A plain    {cfgA['feats']}:  {gpA.kernel_}")
print(f"  D residual {cfgD['feats']}:  {gpD.kernel_}")

### Under the surface (iii): the coordinates lane C adds

The two computed features, drawn over the design space. `logP` is the
calibrated equations' capacity prediction (log newtons): a smooth hill,
strongest at thick-web designs, weakest in the thin-web / tall-web corner.
`stab` pins at exactly 1.0 wherever plain bending governs and drops below 1
only in that same corner — the red contour marks where LTB starts to bite.
Inside it, designs get an extra coordinate that separates them from the rest
of the map, so the GP does not have to smooth across the failure-mode
boundary; a lesson learned there stays there.

There is a catch, and the fit itself reports it (printed below the figure).
Each input feature gets its own length-scale dial, set by maximum likelihood
on 16 beams. On the full dataset the optimizer pushes `logP`'s length scale
to the top of its allowed range (30) — a length scale that large means
"moving along this coordinate barely changes my prediction," i.e. the fit
has effectively switched the feature off. More inputs are not free: every
one adds a dial, and 16 points may not be enough to set the dials well.
Keep that in mind when you read lane C's leave-one-out score.

In [ ]:
bgF = np.linspace(1.25, 7.0, 80); HgF = np.linspace(5.0, 16.0, 80)
BBF, HHF = np.meshgrid(bgF, HgF)
Fq = build_feats(BBF.ravel(), HHF.ravel(), ("b", "H", "logP", "stab"))
LOGP, STAB = Fq[:, 2].reshape(BBF.shape), Fq[:, 3].reshape(BBF.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for a, Z, t in [(ax[0], LOGP, "feature logP — calibrated capacity [log N]"),
                (ax[1], STAB, "feature stab = P_LTB / P_bend")]:
    cf = a.contourf(BBF, HHF, Z, levels=20); fig.colorbar(cf, ax=a)
    a.scatter(df.b, df.H, c="w", edgecolor="k", s=40)
    a.set_xlabel("b [mm]"); a.set_ylabel("H_web [mm]"); a.set_title(t)
ax[1].contour(BBF, HHF, STAB, levels=[0.999], colors="red", linewidths=1.5)
ax[1].annotate("LTB starts to govern", xy=(2.7, 14.6), xytext=(4.1, 12.9),
               color="red", fontsize=9,
               arrowprops=dict(arrowstyle="->", color="red"))
plt.tight_layout(); plt.show()

gpC, fmuC, fsdC, ymC, cfgC = fit_lane(df, "C features")
ls = np.atleast_1d(gpC.kernel_.k2.length_scale)
print("lane C fitted kernel:", gpC.kernel_)
print("  length scales:", ", ".join(f"{n} = {v:.3g}" for n, v in zip(cfgC["feats"], ls)))
switched_off = [n for n, v in zip(cfgC["feats"], ls) if v >= 29.9]
if switched_off:
    print(f"  {', '.join(switched_off)}: at the upper bound (30) — the fit is"
          " effectively ignoring this feature.")

### The scoreboard: leave-one-out cross-validation

The four lanes make different promises. Leave-one-out (LOO) is how we score
them without printing a single new beam. One number per lane, built like
this:

1. Hide beam 1. Fit the lane on the other 15 beams.
2. Predict the hidden beam's str/w and record the miss (predicted − measured).
3. Put it back; hide beam 2 instead. Repeat until every beam has been hidden
   once.
4. The lane's score is the RMSE of the 16 misses, in N/g.

The point of the ritual: **every prediction is made on a beam the model
never saw.** If you instead graded each lane on its own training beams, all
four would score a flattering 0.4–0.5 N/g — near-interpolation that
separates nothing. LOO makes the model earn each point (the honest numbers
land around 2–6 N/g).

What LOO does *not* tell you:

- **Nothing about empty regions.** Every hidden beam still has 15 neighbors
  vouching for it. How a lane behaves far from all data — where your final
  design might sit — is never scored. That is exactly where the lanes differ
  most (look again at the slice figure).
- **16 misses make a noisy RMSE.** One oddball beam can reorder the lanes.
  Look at the per-beam table under the plot, not just the totals.
- **It scores average accuracy, not your decision.** A lane can win the
  table and still be wrong at the one beam you choose to print.

Read it, then choose — you may overrule it if you argue the case.

In [ ]:
loo = {}
for lane in LANES:
    preds = []
    for i in range(len(df)):
        tr = df.drop(df.index[i])                    # hide beam i
        gp_, fmu_, fsd_, ym_, cfg_ = fit_lane(tr, lane)
        sw_hat, _ = predict_sw(gp_, fmu_, fsd_, ym_, df.b.iloc[i], df.H.iloc[i],
                               cfg_["target"], cfg_["feats"])
        preds.append(sw_hat[0])                      # predict it, never having seen it
    loo[lane] = np.array(preds)

meas = df.str_to_weight.values
rmse = {lane: np.sqrt(np.mean((loo[lane] - meas)**2)) for lane in LANES}
print("leave-one-out RMSE in str/w space (lower is better):")
for lane in LANES:
    print(f"  {lane}: {rmse[lane]:.2f} N/g")
print("\nCHECKPOINT: you should arrive at roughly A plain: 2.89 N/g,  B strength: 5.83 N/g,  C features: 3.34 N/g,  D residual: 2.24 N/g.")
print("If your numbers are far off, check your work or talk to a TA.")

LANE_COLORS = {"A plain": "tab:blue", "B strength": "tab:orange",
               "C features": "tab:green", "D residual": "tab:purple"}
lo = min(meas.min(), min(p.min() for p in loo.values())) - 1
hi = max(meas.max(), max(p.max() for p in loo.values())) + 1
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), sharex=True, sharey=True)
for a, lane in zip(axes, LANES):
    a.plot([lo, hi], [lo, hi], "k-", lw=0.8)
    a.scatter(meas, loo[lane], color=LANE_COLORS[lane], s=30, alpha=0.85)
    w = int(np.argmax(np.abs(loo[lane] - meas)))     # flag the worst miss
    a.annotate(f"beam {int(df.beam_id.iloc[w])}", (meas[w], loo[lane][w]),
               textcoords="offset points", xytext=(6, -3), fontsize=8)
    a.set_title(f"{lane} — RMSE {rmse[lane]:.2f}", fontsize=10)
    a.set_xlabel("measured [N/g]")
axes[0].set_ylabel("LOO prediction [N/g]")
fig.suptitle("every point is a prediction of a beam the model never saw",
             y=1.04, fontsize=11)
plt.tight_layout(); plt.show()

err_tab = pd.DataFrame({"beam": df.beam_id.astype(int), "b": df.b, "H": df.H,
                        "measured": meas.round(1)})
for lane in LANES:
    err_tab[lane] = (loo[lane] - meas).round(1)
print("signed LOO misses, predicted − measured [N/g]:")
print(err_tab.to_string(index=False))

### Reading the scoreboard

Connect the numbers to the figures above:

- **Lane A's two worst misses are beams 9 and 12** — (1.75, 15.8), the beam
  that twisted off the test stand, and (1.5, 5.0), a flange-peel failure.
  They are the two most isolated beams on the map. With no close neighbors,
  the plain GP filled in from the rest of the data and landed several N/g
  too high both times. That is the slice figure's lesson wearing real
  numbers.
- **Lane D cuts both of those misses by more than half.** The calibrated
  equations already predict poor str/w in both corners, so the physics
  baseline carries lane D most of the way even with no nearby data. At
  beam 9 the physics alone says 30.9 N/g against a measured 31.0 — it calls
  that corner LTB-governed, and the beam did twist off the stand. (At
  beam 12 the equations are also close, 27.1 vs 28.1 — but note they got
  there without knowing anything about the peel that actually killed it.)
- **Lane B's worst miss is about +15 N/g, on beam 9** — the lightest beam in
  the data. Extrapolating the steep strength trend into that corner, it
  over-predicted the beam's strength by roughly 150 N, and dividing by a
  10-gram mass turned that into the biggest miss on the table.
- **Lane C misses beam 12 worst of all four lanes.** The extra coordinates
  only help where the equations' story is the right story — and beam 12
  died by flange peel, a mode invisible to both `logP` and `stab`.

The table favors lane D on these 16 beams, and the figures say why. They
also say where each lane would betray you. Pick your lane below — the memo
grades the argument, not the lane.

## 3. Rebuild the maps with your lane

Set `CHOICE` and `NOISE_PCT`, refit on all 16 beams, and look at the surfaces
you will decide from.

In [ ]:
CHOICE = "A plain"      # <<< your lane
NOISE_PCT = 3           # <<< your assumed noise, percent

gpF, fmuF, fsdF, ymF, cfgF = fit_lane(df, CHOICE, alpha=(NOISE_PCT/100)**2)
bg = np.linspace(1.25, 7.0, 60); Hg = np.linspace(5.0, 16.0, 60)
BB, HH = np.meshgrid(bg, Hg)
SWg, SDg = predict_sw(gpF, fmuF, fsdF, ymF, BB.ravel(), HH.ravel(),
                      cfgF["target"], cfgF["feats"])
MU, STD = SWg.reshape(BB.shape), SDg.reshape(BB.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for a, Z, t in [(ax[0], MU, f"mean str/w — {CHOICE}"), (ax[1], STD, "uncertainty (log units)")]:
    cf = a.contourf(BB, HH, Z, levels=20); fig.colorbar(cf, ax=a)
    a.scatter(df.b, df.H, c="w", edgecolor="k", s=40)
    a.scatter(df.b.iloc[-2:], df.H.iloc[-2:], c="red", marker="*", s=170,
              label="the two class beams")
    a.set_xlabel("b [mm]"); a.set_ylabel("H_web [mm]"); a.set_title(t); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 4. Choose your beam *(you write this — graded on the memo, not a checkpoint)*

Combine what you have: `MU`, `STD`, the calibrated physics (`capacity`), and
the fracture notes. Standard moves: exploit the mean; MUI-style explore; veto
regions the notes make you distrust (which beams peeled apart, and where?).

For orientation only: the *default* recipe — lane A, 3% noise, MUI ψ = 1 —
lands at **(b ≈ 1.25, H_web ≈ 13.20)**. You are free to submit that; you
are also free to beat it. Say why either way.

In [ ]:
psi = 1.0                                  # <<< your dial
score = np.log(MU) + psi*STD               # <<< your rule (this is MUI in log space)
i = np.unravel_index(np.argmax(score), score.shape)
b_final, H_final = float(BB[i]), float(HH[i])
print(f"FINAL DESIGN:  b = {b_final:.2f} mm,  H_web = {H_final:.2f} mm")
print(f"  model mean {MU[i]:.1f} N/g,  sigma {STD[i]:.3f},  "
      f"calibrated physics {capacity(b_final, H_final, SY_CAL, K_CAL, CS_CAL)/mass_g(b_final, H_final):.1f} N/g")
print(f"  physics mode there: {gov_mode(b_final, H_final, SY_CAL, K_CAL, CS_CAL)}")

## Memo (the graded part)

1. **The two class beams.** One model over-promised by 24%, the other
   under-promised by 4%. What does each miss tell you about each model — and
   which kind of miss would you rather design against?
2. **Your lane.** What did the LOO table say, and did you follow it? If you
   chose C or D: what does the physics buy you *between* the data points?
3. **Risk.** Are you exploiting or exploring, and what is the failure mode of
   your choice — the beam being weak, or your model being sure of the wrong
   thing?
4. **Limits.** The thin-web beams in your data peeled at the flange–web joint
   at loads no yield formula flagged. Where would that mode bite your design?